# CMS Medicare Part D - Build Database

This notebook loads the CMS Medicare Part D Prescribers by Geography and Drug (2023) CSV
into a local SQLite database for analysis.

**Source:** CMS Medicare Part D Prescribers by Geography and Drug, 2023  
**Rows:** ~116,000 (one row per drug per geographic unit)  
**Geographic levels:** National and State  

Run this notebook once before any analysis notebooks.

In [1]:
import sqlite3
import pandas as pd

csv_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\CMS\data\Medicare_Part_D_Prescribers_by_Geography_and_Drug_2023.csv"
db_path  = r"C:\Users\palla\OneDrive\Documents\Coding Projects\CMS\database\cms.db"

conn = sqlite3.connect(db_path)

## Step 1: Load CSV into pandas

Read the raw CSV. Some numeric columns contain suppression flags (empty strings)
where CMS withheld values to protect beneficiary privacy â€” we set these to NaN.

In [2]:
df = pd.read_csv(csv_path, dtype=str)  # read everything as string first to preserve suppression flags

print(f"Rows: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head(3)

Rows: 115,936
Columns: ['Prscrbr_Geo_Lvl', 'Prscrbr_Geo_Cd', 'Prscrbr_Geo_Desc', 'Brnd_Name', 'Gnrc_Name', 'Tot_Prscrbrs', 'Tot_Clms', 'Tot_30day_Fills', 'Tot_Drug_Cst', 'Tot_Benes', 'GE65_Sprsn_Flag', 'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Bene_Sprsn_Flag', 'GE65_Tot_Benes', 'LIS_Bene_Cst_Shr', 'NonLIS_Bene_Cst_Shr', 'Opioid_Drug_Flag', 'Opioid_LA_Drug_Flag', 'Antbtc_Drug_Flag', 'Antpsyct_Drug_Flag']


,Prscrbr_Geo_Lvl,Prscrbr_Geo_Cd,Prscrbr_Geo_Desc,Brnd_Name,Gnrc_Name,Tot_Prscrbrs,Tot_Clms,Tot_30day_Fills,Tot_Drug_Cst,Tot_Benes,...,GE65_Tot_30day_Fills,GE65_Tot_Drug_Cst,GE65_Bene_Sprsn_Flag,GE65_Tot_Benes,LIS_Bene_Cst_Shr,NonLIS_Bene_Cst_Shr,Opioid_Drug_Flag,Opioid_LA_Drug_Flag,Antbtc_Drug_Flag,Antpsyct_Drug_Flag
0,National,NaN,National,1st Tier Unifine Pentips,"Pen Needle, Diabetic",691,1613,2874.2,44355.04,699,...,2375.6,35803.14,NaN,565,992.77,7093.55,N,N,N,N
1,National,NaN,National,1st Tier Unifine Pentips Plus,"Pen Needle, Diabetic",1167,3269,5592,97951.18,1267,...,4545,76544.14,NaN,1035,1899.59,10392.03,N,N,N,N
2,National,NaN,National,Abacavir,Abacavir Sulfate,2617,19634,25152.1,5290175.88,2807,...,15353.6,3042569,NaN,1662,10643.29,152140.95,N,N,N,N


## Step 2: Clean and cast column types

Convert numeric columns from string to float, coercing suppressed values to NaN.
Flag columns (Opioid, Antibiotic, etc.) stay as strings ('Y'/'N').

In [3]:
numeric_cols = [
    'Tot_Prscrbrs', 'Tot_Clms', 'Tot_30day_Fills', 'Tot_Drug_Cst', 'Tot_Benes',
    'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Tot_Benes',
    'LIS_Bene_Cst_Shr', 'NonLIS_Bene_Cst_Shr'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')  # empty strings and suppression markers become NaN

print("Null counts per numeric column:")
print(df[numeric_cols].isnull().sum())

Null counts per numeric column:
Tot_Prscrbrs                0
Tot_Clms                    0
Tot_30day_Fills             0
Tot_Drug_Cst                0
Tot_Benes               21596
GE65_Tot_Clms           21063
GE65_Tot_30day_Fills    21063
GE65_Tot_Drug_Cst       21063
GE65_Tot_Benes          48848
LIS_Bene_Cst_Shr            0
NonLIS_Bene_Cst_Shr         0
dtype: int64


## Step 3: Write to SQLite

Load the cleaned dataframe into a table called `part_d_geo`. 
This replaces the table each time the notebook is run.

In [4]:
df.to_sql('part_d_geo', conn, if_exists='replace', index=False)

row_count = conn.execute("SELECT COUNT(*) FROM part_d_geo").fetchone()[0]
print(f"Rows loaded into part_d_geo: {row_count:,}")

Rows loaded into part_d_geo: 115,936


## Step 4: Verify

Confirm geographic levels and row distribution.

In [5]:
geo_levels = pd.read_sql_query("""
    SELECT Prscrbr_Geo_Lvl, COUNT(*) AS rows
    FROM part_d_geo
    GROUP BY Prscrbr_Geo_Lvl
""", conn)

print(geo_levels)

# Preview a few rows of state-level data
sample = pd.read_sql_query("""
    SELECT Prscrbr_Geo_Desc, Brnd_Name, Gnrc_Name, Tot_Prscrbrs, Tot_Clms, Tot_Drug_Cst
    FROM part_d_geo
    WHERE Prscrbr_Geo_Lvl = 'State'
    LIMIT 5
""", conn)

print(sample)

  Prscrbr_Geo_Lvl    rows
0        National    3607
1           State  112329
  Prscrbr_Geo_Desc                      Brnd_Name  \
0              NaN                  Trazodone Hcl   
1          Alabama  1st Tier Unifine Pentips Plus   
2          Alabama                       Abacavir   
3          Alabama            Abacavir-Lamivudine   
4          Alabama                        Abilify   

                     Gnrc_Name  Tot_Prscrbrs  Tot_Clms  Tot_Drug_Cst  
0                Trazodone Hcl             1        11         77.45  
1         Pen Needle, Diabetic            13        25        838.55  
2             Abacavir Sulfate            44       364      79474.16  
3  Abacavir Sulfate/Lamivudine            23       135      44676.00  
4                 Aripiprazole            10        58      53531.77  


## Step 5: Load Medicare Enrollment Data

Load state-level Medicare enrollment counts for 2023. This is used to normalize
prescribing counts by Part D enrollment, producing per-capita metrics by state.

The source file contains annual enrollment by state across all years.
We filter to 2023 annual state-level rows and keep only the columns needed for analysis.

In [ ]:
enroll_path = r"C:\Users\palla\OneDrive\Documents\Coding Projects\CMS\data\Medicare_Monthly_Enrollment_Dec_2025.csv"

enroll_raw = pd.read_csv(enroll_path, dtype=str)

# Filter to 2023 annual state-level rows only
enroll = enroll_raw[
    (enroll_raw['YEAR'] == '2023') &
    (enroll_raw['MONTH'] == 'Year') &
    (enroll_raw['BENE_GEO_LVL'] == 'State')
].copy()

# Keep only columns needed for joining and normalization
enroll = enroll[[
    'BENE_STATE_ABRVTN',          # state abbreviation (e.g. CA) â€” join key
    'BENE_STATE_DESC',            # full state name
    'BENE_FIPS_CD',               # FIPS code
    'TOT_BENES',                  # total Medicare beneficiaries
    'PRSCRPTN_DRUG_TOT_BENES'    # total Part D enrollees â€” used for normalization
]].copy()

# Cast counts to numeric
enroll['TOT_BENES'] = pd.to_numeric(enroll['TOT_BENES'], errors='coerce')
enroll['PRSCRPTN_DRUG_TOT_BENES'] = pd.to_numeric(enroll['PRSCRPTN_DRUG_TOT_BENES'], errors='coerce')

print(f"States loaded: {len(enroll)}")
enroll.head()

## Step 6: Write Enrollment Table to Database

In [ ]:
enroll.to_sql('state_enrollment', conn, if_exists='replace', index=False)

row_count = conn.execute("SELECT COUNT(*) FROM state_enrollment").fetchone()[0]
print(f"Rows loaded into state_enrollment: {row_count}")

# Verify â€” check top states by Part D enrollment
pd.read_sql_query("""
    SELECT BENE_STATE_ABRVTN, BENE_STATE_DESC, TOT_BENES, PRSCRPTN_DRUG_TOT_BENES
    FROM state_enrollment
    ORDER BY PRSCRPTN_DRUG_TOT_BENES DESC
    LIMIT 10
""", conn)